# Pakistan Earthquake Damage-Risk Model

This notebook trains, evaluates and documents the earthquake damage-risk model used by
DisasterGuard. It is the reproducible version of `backend/ml/train_earthquake.py`.

**Algorithm:** `sklearn.ensemble.RandomForestClassifier` only.

**Target:** `damage_occurred` (0 = no reported damage, 1 = damage reported)

**Dataset:** `datasets/pakistan_earthquake_2020_2025.csv` (643 events, 30 positive events)

The notebook covers:

1. Exploratory data analysis
2. Feature engineering and feature-set selection (RandomForest only)
3. Hyperparameter tuning (RandomForest only)
4. Repeated stratified cross-validation, temporal validation and a chronological holdout
5. Precision, recall, F1, F2, ROC-AUC, PR-AUC and confusion matrices
6. Explainability and integration with the FastAPI service


## 1. Setup

The notebook adds the repository root and `backend/ml` to `sys.path` so that the same
feature-engineering and training functions used by the production pipeline can be
imported here.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "datasets" / "pakistan_earthquake_2020_2025.csv").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate the repository root")
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "backend" / "ml"))
sys.path.insert(0, str(ROOT / "backend"))

from earthquake_features import (
    CATEGORICAL_FEATURES,
    CONTEXT_FEATURES,
    FEATURE_SETS,
    RAW_NUMERIC_FEATURES,
    STATIC_ENGINEERED_FEATURES,
    TARGET,
    engineer_features,
    load_data,
)
from train_earthquake import run_training

print("Repository root:", ROOT)
print("Feature sets:", {name: len(cols) for name, cols in FEATURE_SETS.items()})


C:\Users\moizs\AppData\Local\Temp\ipykernel_8208\3973913758.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Repository root: C:\Users\moizs\Downloads\AI-Based-Multi-Disaster-Risk-Prediction-And-Mapping-System
Feature sets: {'raw': 6, 'static': 16, 'context': 26}


## 2. Exploratory data analysis

The dataset contains one row per earthquake event. `deaths` is deliberately **excluded**
from modelling because it is an outcome of the same event and would leak the target.


In [2]:
DATA_PATH = ROOT / "datasets" / "pakistan_earthquake_2020_2025.csv"
df = load_data(DATA_PATH)

print("Shape:", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Positive rate: %.2f%%" % (100 * df[TARGET].mean()))
display(df.head())
display(df["damage_occurred"].value_counts().rename(index={0: "No damage", 1: "Damage"}).to_frame("count"))
display(df.groupby("province")["damage_occurred"].agg(["sum", "count", "mean"]).sort_values("sum", ascending=False))


Shape: (643, 9)
Date range: 2020-01-01 to 2025-12-30
Positive rate: 4.67%


,date,latitude,longitude,magnitude,depth_km,damage_occurred,city,province,deaths
0,2020-01-01,36.2504,74.2021,3.0,34.5,0,Gilgit,GB,0
1,2020-01-06,27.7105,67.1090,1.7,32.6,0,Khuzdar,Balochistan,0
2,2020-01-08,33.1500,73.7500,4.3,16.5,0,Mirpur,AJK,0
3,2020-01-11,34.8159,73.6909,2.4,23.0,0,Muzaffarabad,AJK,0
4,2020-01-15,30.1798,66.9750,3.7,20.5,0,Quetta,Balochistan,0


,count
damage_occurred,
No damage,613
Damage,30


,sum,count,mean
province,,,
Balochistan,18,213,0.084507
AJK,10,117,0.085470
GB,1,73,0.013699
KPK,1,117,0.008547
ICT,0,59,0.000000
Punjab,0,64,0.000000


In [3]:
display(df[["magnitude", "depth_km", "latitude", "longitude", "deaths"]].describe().T)
print("\nCorrelation with damage_occurred:")
display(df[["magnitude", "depth_km", "damage_occurred", "deaths"]].corr()[["damage_occurred"]])
print("\nDamaging events:")
display(df[df[TARGET] == 1][["date", "magnitude", "depth_km", "city", "province", "deaths"]].to_string(index=False))


,count,mean,std,min,25%,50%,75%,max
magnitude,643.0,2.972628,0.945623,1.5000,2.3000,2.9000,3.40000,6.2000
depth_km,643.0,31.239347,18.549678,4.5000,15.4000,25.2000,46.85000,69.8000
latitude,643.0,32.737107,2.571452,27.3237,30.2227,33.7037,34.42270,36.4001
longitude,643.0,71.132180,2.900235,66.1441,67.8516,72.6085,73.41165,74.7952
deaths,643.0,0.242613,1.702185,0.0000,0.0000,0.0000,0.00000,26.0000



Correlation with damage_occurred:


,damage_occurred
magnitude,0.509000
depth_km,-0.175998
damage_occurred,1.000000
deaths,0.644784



Damaging events:


'      date  magnitude  depth_km         city    province  deaths\n2020-04-18        5.1      12.7 Muzaffarabad         AJK       2\n2020-07-14        4.9      34.7       Khyber         KPK       1\n2020-08-22        5.1      19.4       Chaman Balochistan       3\n2020-11-19        5.5      16.6       Harnai Balochistan      10\n2021-04-05        5.1      15.6       Mirpur         AJK       2\n2021-06-10        5.9      14.0       Harnai Balochistan      20\n2021-07-22        4.7      17.1       Chaman Balochistan       1\n2021-09-18        5.4      12.9 Muzaffarabad         AJK       5\n2021-10-07        6.2      10.8       Harnai Balochistan      26\n2021-10-08        5.1      11.9       Harnai Balochistan       3\n2022-04-07        5.0      13.7 Muzaffarabad         AJK       3\n2022-06-21        5.0      17.2       Ziarat Balochistan       2\n2022-08-25        4.8      21.4       Chaman Balochistan       1\n2022-10-05        5.4      11.5       Mirpur         AJK       8\n2022-11-1

### What the EDA shows

- Only **30 of 643 events (4.67%)** caused reported damage, so accuracy alone is misleading.
- Damaging events are concentrated in shallow-to-intermediate depth and mostly have
  magnitude >= 4.7. This makes the labels highly separable and explains the very high
  scores, while also showing that the dataset is small and close to a rule-based label.
- Balochistan and AJK dominate the positive class; many Pakistani cities are absent.
- `deaths` correlates strongly with the target and is a leakage feature, so it is dropped.


## 3. Feature engineering and feature selection

Three candidate feature sets are compared with RandomForest:

| Set | Contents |
| --- | --- |
| `raw` | latitude, longitude, magnitude, depth, province, city |
| `static` | raw + magnitude squared, shallow-depth flags, magnitude/depth ratio, cyclical month/day-of-year, approximate fault-belt distance |
| `context` | static + past 7/30/90-day event counts, past maximum magnitude, past mean depth, local density within 100 km, time and distance to the previous event |

Context features are computed strictly from **earlier** events (strictly earlier dates;
same-day events are excluded). No target-derived historical feature (for example previous
damage counts) is used, which avoids label leakage during validation.


In [4]:
engineered = engineer_features(df, include_context=True)
print("Raw numeric:", RAW_NUMERIC_FEATURES)
print("Static engineered:", STATIC_ENGINEERED_FEATURES)
print("Historical/context:", CONTEXT_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
display(engineered[RAW_NUMERIC_FEATURES + STATIC_ENGINEERED_FEATURES + CONTEXT_FEATURES].head(8))


Raw numeric: ['latitude', 'longitude', 'magnitude', 'depth_km']
Static engineered: ['magnitude_squared', 'is_shallow', 'is_very_shallow', 'mag_depth_ratio', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'fault_distance_km', 'near_fault']
Historical/context: ['prior_count_7d', 'prior_count_30d', 'prior_count_90d', 'prior_max_mag_30d', 'prior_max_mag_90d', 'prior_mean_depth_30d', 'prior_count_100km_90d', 'min_distance_past_km', 'days_since_previous', 'distance_to_previous_km']
Categorical: ['province', 'city']


,latitude,longitude,magnitude,depth_km,magnitude_squared,is_shallow,is_very_shallow,mag_depth_ratio,month_sin,month_cos,...,prior_count_7d,prior_count_30d,prior_count_90d,prior_max_mag_30d,prior_max_mag_90d,prior_mean_depth_30d,prior_count_100km_90d,min_distance_past_km,days_since_previous,distance_to_previous_km
0,36.2504,74.2021,3.0,34.5,9.00,0,0,0.086957,0.5,0.866025,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,9999.000000,9999.0,9999.000000
1,27.7105,67.1090,1.7,32.6,2.89,0,0,0.052147,0.5,0.866025,...,1.0,1.0,1.0,3.0,3.0,34.500000,0.0,1160.751749,5.0,1160.751749
2,33.1500,73.7500,4.3,16.5,18.49,1,0,0.260606,0.5,0.866025,...,2.0,2.0,2.0,3.0,3.0,33.550000,0.0,347.215866,2.0,877.790089
3,34.8159,73.6909,2.4,23.0,5.76,1,0,0.104348,0.5,0.866025,...,2.0,3.0,3.0,4.3,4.3,27.866667,0.0,166.080198,3.0,185.319748
4,30.1798,66.9750,3.7,20.5,13.69,1,0,0.180488,0.5,0.866025,...,2.0,4.0,4.0,4.3,4.3,26.650000,0.0,274.882958,4.0,813.532863
5,29.6802,67.0704,1.8,25.1,3.24,1,0,0.071713,0.5,0.866025,...,2.0,5.0,5.0,4.3,4.3,25.420000,1.0,56.308519,1.0,56.308519
6,33.4746,72.8329,2.8,49.5,7.84,0,0,0.056566,0.5,0.866025,...,2.0,6.0,6.0,4.3,4.3,25.366667,1.0,92.548875,5.0,689.713380
7,34.0077,71.5711,3.1,53.7,9.61,0,0,0.057728,0.5,0.866025,...,1.0,7.0,7.0,4.3,4.3,28.814286,0.0,130.865742,5.0,130.865742


## 4. Train the model

The training function performs the full experiment with RandomForest only:

1. 5-fold stratified CV for 3 feature sets (15 fits)
2. Hyperparameter tuning with GridSearchCV (36 parameter sets x 5 folds = 180 fits)
3. Out-of-fold threshold search on training data (5 fits)
4. Repeated 5-fold x 3 CV for stability (15 fits)
5. 5-fold TimeSeriesSplit temporal validation (5 fits)
6. Chronological 80/20 holdout (1 fit)

Total: **221 model fits** with fixed random seeds. The deployment model is trained on
the first 80% of chronological data only; no refit on all data. Artefacts are written to
`backend/ml/artifacts/`.


In [5]:
metadata = run_training(DATA_PATH, ROOT / "backend" / "ml" / "artifacts")
metadata["training_runs"]


Dataset: 643 rows | 30 positives | 613 negatives (4.67%)
Train: 514 rows | 24 positives
Test:  129 rows | 6 positives

Stage 1 - feature-set comparison with RandomForest


  raw      random_forest       PR-AUC=0.9853 F1=0.9314 Recall=0.9100 Precision=0.9600


  static   random_forest       PR-AUC=0.9248 F1=0.7854 Recall=0.7400 Precision=0.8433


  context  random_forest       PR-AUC=0.9886 F1=0.8810 Recall=0.8300 Precision=0.9667

Selected feature set: context (mean PR-AUC 0.9886)

Stage 2 - hyperparameter tuning (5-fold stratified)


  random_forest       best CV PR-AUC=0.9933 params={'model__max_depth': None, 'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__n_estimators': 300}

Stage 3 - repeated stratified CV (5 folds x 3 repeats)


  random_forest       PR-AUC=0.9824 +/- 0.0237 | F1=0.8756 | Recall=0.8767 | Precision=0.9071

Stage 4 - temporal validation (TimeSeriesSplit, 5 folds)


... (2 outputs left out)

  Holdout @0.50: PR-AUC=1.0000 P=1.0000 R=0.6667 F1=0.8000
  Holdout @0.19 (best F1): P=1.0000 R=1.0000 F1=1.0000
  Holdout @0.19 (best F2/safety): P=1.0000 R=1.0000 F1=1.0000
  OOF best F1: 0.8889 at threshold 0.19
  OOF best F2: 0.9524 at threshold 0.19

Stage 6 - permutation importance on chronological holdout


                feature  importance_mean  importance_std
      magnitude_squared     2.564159e-01    9.622569e-02
              magnitude     2.244643e-01    7.025752e-02
             is_shallow     2.775558e-17    4.807407e-17
      prior_max_mag_90d     2.775558e-17    4.807407e-17
        prior_count_30d     5.551115e-18    2.419675e-17
         prior_count_7d     5.551115e-18    2.419675e-17
               latitude     0.000000e+00    0.000000e+00
               province     0.000000e+00    0.000000e+00
distance_to_previous_km     0.000000e+00    0.000000e+00
    days_since_previous     0.000000e+00    0.000000e+00



Saved artifacts to C:\Users\moizs\Downloads\AI-Based-Multi-Disaster-Risk-Prediction-And-Mapping-System\backend\ml\artifacts
Total model fits: 221
Training time: 106.4s


{'feature_set_comparison_fits': 15,
 'hyperparameter_candidate_sets': 36,
 'hyperparameter_fits': 180,
 'oof_threshold_fits': 5,
 'repeated_cv_fits': 15,
 'temporal_cv_fits': 5,
 'holdout_fits': 1,
 'final_refit_fits': 0,
 'total_fits': 221}

## 5. Results

### 5.1 Feature-set comparison

Mean 5-fold stratified CV scores for the default RandomForest. PR-AUC is the primary
metric because the positive class is rare.


In [6]:
comparison = pd.DataFrame(metadata["feature_set_comparison"])
display(
    comparison[
        ["feature_set", "model", "pr_auc", "precision", "recall", "f1", "roc_auc", "accuracy"]
    ].sort_values(["feature_set", "pr_auc"], ascending=[True, False]).round(4)
)

print("Mean PR-AUC by feature set:")
display(comparison.groupby("feature_set")["pr_auc"].mean().sort_values(ascending=False).round(4))


,feature_set,model,pr_auc,precision,recall,f1,roc_auc,accuracy
2,context,random_forest,0.9886,0.9667,0.83,0.8810,0.9992,0.9903
0,raw,random_forest,0.9853,0.9600,0.91,0.9314,0.9992,0.9942
1,static,random_forest,0.9248,0.8433,0.74,0.7854,0.9965,0.9825


Mean PR-AUC by feature set:


feature_set
context    0.9886
raw        0.9853
static     0.9248
Name: pr_auc, dtype: float64

### 5.2 Hyperparameter tuning

RandomForest was tuned with GridSearchCV (scoring `average_precision`, 5-fold stratified CV)
on the selected feature set using training data only.


In [7]:
tuning = pd.DataFrame(metadata["hyperparameter_tuning"])
display(tuning[["model", "best_cv_pr_auc", "candidate_parameter_sets", "cv_folds", "best_params"]].round(4))


,model,best_cv_pr_auc,candidate_parameter_sets,cv_folds,best_params
0,random_forest,0.9933,36,5,"{""model__max_depth"": null, ""model__max_feature..."


### 5.3 Repeated stratified CV (5 folds x 3 repeats)

This is the most stable estimate on the training partition.


In [8]:
repeated = pd.DataFrame(metadata["repeated_cv"])
display(
    repeated[
        ["model", "pr_auc", "pr_auc_std", "precision", "recall", "f1", "roc_auc", "accuracy"]
    ].round(4)
)


,model,pr_auc,pr_auc_std,precision,recall,f1,roc_auc,accuracy
0,random_forest,0.9824,0.0237,0.9071,0.8767,0.8756,0.999,0.9883


### 5.4 Temporal validation (TimeSeriesSplit, 5 folds)

Events are ordered by date and each fold trains on the past and tests on the future. This
checks whether the model still works when it cannot see later events.


In [9]:
temporal = pd.DataFrame(metadata["temporal_cv"])
display(
    temporal[
        ["model", "pr_auc", "precision", "recall", "f1", "roc_auc", "accuracy"]
    ].round(4)
)


,model,pr_auc,precision,recall,f1,roc_auc,accuracy
0,random_forest,0.9394,0.8667,0.7833,0.819,0.9955,0.9859


### 5.5 Chronological holdout (last 20%)

The final 129 events (6 positives) were never used for tuning or threshold selection.
The deployment threshold comes from out-of-fold predictions on the training partition.


In [10]:
evaluation = metadata["evaluation"]
rows = []
for name in ["holdout_default_threshold", "holdout_f1_optimal_threshold", "holdout_deployment_threshold"]:
    result = evaluation[name]
    rows.append(
        {
            "evaluation": name,
            "threshold": result["threshold"],
            "accuracy": result["accuracy"],
            "balanced_accuracy": result["balanced_accuracy"],
            "precision": result["precision"],
            "recall": result["recall"],
            "f1": result["f1"],
            "roc_auc": result["roc_auc"],
            "pr_auc": result["pr_auc"],
            "TP": result["tp"],
            "FP": result["fp"],
            "FN": result["fn"],
            "TN": result["tn"],
        }
    )
display(pd.DataFrame(rows).round(4))
print("Deployment threshold is chosen with F2 on out-of-fold predictions so that recall is favoured over precision.")


,evaluation,threshold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,TP,FP,FN,TN
0,holdout_default_threshold,0.50,0.9845,0.8333,1.0,0.6667,0.8,1.0,1.0,4,0,2,123
1,holdout_f1_optimal_threshold,0.19,1.0000,1.0000,1.0,1.0000,1.0,1.0,1.0,6,0,0,123
2,holdout_deployment_threshold,0.19,1.0000,1.0000,1.0,1.0000,1.0,1.0,1.0,6,0,0,123


Deployment threshold is chosen with F2 on out-of-fold predictions so that recall is favoured over precision.


In [11]:
deployment = evaluation["holdout_deployment_threshold"]
cm = pd.DataFrame(
    deployment["confusion_matrix"],
    index=["Actual no damage", "Actual damage"],
    columns=["Predicted no damage", "Predicted damage"],
)
display(cm)
print("Classification report at the deployment threshold:")
display(pd.read_csv(ROOT / "backend" / "ml" / "artifacts" / "holdout_classification_report.csv", index_col=0).round(4))


,Predicted no damage,Predicted damage
Actual no damage,123,0
Actual damage,0,6


Classification report at the deployment threshold:


,precision,recall,f1-score,support
0,1.0,1.0,1.0,123.0
1,1.0,1.0,1.0,6.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,129.0
weighted avg,1.0,1.0,1.0,129.0


### 5.6 What the model relies on

Tree-model feature importance is shown first, followed by permutation importance on the
chronological holdout. The holdout has only six positives, so permutation importance is
noisy; it is reported for interpretation rather than model selection.


In [12]:
importance = pd.read_csv(ROOT / "backend" / "ml" / "artifacts" / "feature_importance.csv")
display(importance.head(15).round(4))

permutation = pd.read_csv(ROOT / "backend" / "ml" / "artifacts" / "permutation_importance.csv")
display(permutation.head(15).round(4))


,feature,importance
0,numeric__magnitude_squared,0.4092
1,numeric__magnitude,0.4033
2,numeric__mag_depth_ratio,0.0673
3,numeric__min_distance_past_km,0.0473
4,numeric__depth_km,0.0170
5,numeric__distance_to_previous_km,0.0104
6,numeric__prior_max_mag_30d,0.0103
7,numeric__prior_mean_depth_30d,0.0050
8,categorical__city_Quetta,0.0043
9,numeric__prior_count_7d,0.0041


,feature,importance_mean,importance_std
0,magnitude_squared,0.2564,0.0962
1,magnitude,0.2245,0.0703
2,is_shallow,0.0000,0.0000
3,prior_max_mag_90d,0.0000,0.0000
4,prior_count_30d,0.0000,0.0000
5,prior_count_7d,0.0000,0.0000
6,latitude,0.0000,0.0000
7,province,0.0000,0.0000
8,distance_to_previous_km,0.0000,0.0000
9,days_since_previous,0.0000,0.0000


In [13]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
top = importance.head(12).iloc[::-1]
ax.barh(top["feature"], top["importance"], color="#7b2cbf")
ax.set_xlabel("Feature importance")
ax.set_title("Top features of the final earthquake model")
plt.tight_layout()
plt.show()


<Figure size 900x500 with 1 Axes>

## 6. Integration with the FastAPI backend

The trained bundle contains the fitted pipeline, the deployment threshold, the reference
event history and all metadata. The service loads it lazily and exposes:

- `POST /api/earthquake/predict`
- `GET /api/earthquake/risk-by-city?city=Harnai, Balochistan`
- `GET /api/earthquake/events?city=Islamabad&limit=10`
- `GET /api/earthquake/model-info`

The next cell calls the service directly, without HTTP, to show the response shape.


In [14]:
from services.earthquake_service import predict_event

sample = predict_event(
    {
        "magnitude": 5.8,
        "depth_km": 12.9,
        "latitude": 30.0,
        "longitude": 67.0,
        "province": "Balochistan",
        "city": "Harnai",
        "date": "2025-03-03",
    }
)
print(json.dumps({k: sample[k] for k in ["model", "prediction", "factors"]}, indent=2))


{
  "model": {
    "name": "random_forest",
    "version": "3.0.0",
    "feature_set": "context",
    "threshold": 0.19
  },
  "prediction": {
    "damage_probability": 0.5467,
    "damage_predicted": true,
    "risk_score": 55,
    "risk_level": "high",
    "confidence_band": "medium",
    "threshold_used": 0.19
  },
  "factors": [
    {
      "key": "magnitude",
      "label": "Magnitude Activity",
      "value": "5.8 ML",
      "score": 0.9149,
      "color": "#ff1744"
    },
    {
      "key": "depth",
      "label": "Shallow Depth",
      "value": "12.9 km",
      "score": 0.8157,
      "color": "#ff1744"
    },
    {
      "key": "fault",
      "label": "Fault Belt Proximity",
      "value": "120 km",
      "score": 0.5985,
      "color": "#ff6d00"
    },
    {
      "key": "activity",
      "label": "Regional Seismicity (90d)",
      "value": "28 events",
      "score": 0.9742,
      "color": "#ff1744"
    },
    {
      "key": "local",
      "label": "Local Density (100 km / 90

## 7. Challenges, limitations and next steps

### Main challenges

1. **Small dataset:** 643 events and only 30 positives. CV metrics have wide uncertainty,
   and the holdout has only six positives.
2. **Severe class imbalance:** 4.67% positives. Accuracy is misleading; PR-AUC, recall,
   F1, class weights and threshold tuning are used instead.
3. **Near-deterministic labels:** most damaging events are magnitude >= 4.7 and shallow.
   The very high scores partly reflect this rule-like pattern, not necessarily real-world
   predictive power.
4. **Leakage control:** `deaths` was removed. Historical features use only earlier events.
5. **Missing real risk drivers:** fault geometry, peak ground acceleration, intensity
   (MMI), soil type, building stock, population density and epicentre-to-city distance are
   not in the dataset.
6. **Geographic imbalance:** Balochistan and AJK dominate; only 15 cities appear.
7. **Threshold trade-off:** deployment threshold is derived from out-of-fold F2 on the
   training partition so recall is favoured over precision; false negatives are costlier
   for disaster alerts.
8. **Static dataset:** the model is not yet connected to live USGS feeds.

### Recommended next steps

- Collect more events and real damage labels (NDMA, USGS, EMSC, PMD).
- Add PGA/MMI, Vs30/soil, building vulnerability and population exposure.
- Validate on a truly unseen future year.
- Calibrate probabilities (Platt/isotonic) and review the alert threshold with domain experts.
- Add live USGS polling and scheduled retraining.

### Important

This is an earthquake damage-risk prototype, not a validated earthquake early-warning
system.
